In [1]:
import sys
sys.path.append('../')
import scipy.io as sio
import mat73
import pandas as pd
import torch
import numpy as np
import torch.optim as optim
import torch.nn
import sklearn
import sklearn.metrics
import matplotlib.pyplot as plt
from alive_progress import alive_bar
from  utils.my_classes import dataset 
from torch.utils.data import DataLoader
import utils.DNN_functions as DNN_functions
import scipy
import random
import utils.AMSloss
from speechbrain.pretrained import SpeakerRecognition
import pickle
import os
from utils.my_classes import dataset
import utils.eval_metrics as eval_metrics
import copy
from speechbrain.pretrained import SpeakerRecognition

#To get my GPU device - GTX 4070 :)
seed = 42  # You can choose any integer value as the seed
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu');

if torch.cuda.is_available():
    print(torch.cuda.device_count())
    print(torch.cuda.device(0))
    print(torch.cuda.get_device_name(0))
    print(device)

The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.
c:\Users\avish\OneDrive\Desktop\thesis_research\avishai_work\env\lib\site-packages\speechbrain\utils\torch_audio_backend.py:22: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
c:\Users\avish\OneDrive\Desktop\thesis_research\avishai_work\env\lib\site-packages\torch\onnx\_internal\_beartype.py:35: UserWarning: unhashable type: 'list'
  warnings.warn(f"{e}")
The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.


1
NVIDIA GeForce RTX 4070
cuda


c:\Users\avish\OneDrive\Desktop\thesis_research\avishai_work\env\lib\site-packages\speechbrain\utils\torch_audio_backend.py:22: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


In [2]:
from ASV_utils.tdcf_functions import compute_t_dcf


import pickle
with open('./dcf_and_tdcf/dcf_avg_score/pmf_both_tdcf/asv_protocol/objects_asv_and_cm_time_embeddings_eval_avg_score_both_asv_protocol.pkl', 'rb') as f:
    results_eval = pickle.load(f)


In [3]:
from ASV_utils.data_loading import *



models_folder = "ECAPA_TDNN/inference_models/models_both_not_normalize/"

data_path_male = "Data/pmf_both/not_normalize/male/"

data_path_female = "Data/pmf_both/not_normalize/female/"

male_embedded_groups_1_1,male_embedded_groups_1_2,male_embedded_groups_1_3,male_chosen_labels_1_1_is_spoofed,male_chosen_labels_2_1_is_spoofed,male_chosen_labels_3_1_is_spoofed,male_chosen_labels_numeric_1_1,male_chosen_labels_numeric_2_1,male_chosen_labels_numeric_3_1,male_chosen_labels_1_1_attack_logical,male_chosen_labels_2_1_attack_logical,male_chosen_labels_3_1_attack_logical,male_chosen_labels_1_1_name,male_chosen_labels_2_1_name,male_chosen_labels_3_1_name,male_chosen_labels_1_1_speaker_id,male_chosen_labels_2_1_speaker_id,male_chosen_labels_3_1_speaker_id,male_chosen_labels_1_1_sex,male_chosen_labels_2_1_sex,male_chosen_labels_3_1_sex  = load_data_male(data_path_male)

female_embedded_groups_1_1,female_embedded_groups_1_2,female_embedded_groups_1_3,female_chosen_labels_1_1_is_spoofed,female_chosen_labels_2_1_is_spoofed,female_chosen_labels_3_1_is_spoofed,female_chosen_labels_numeric_1_1,female_chosen_labels_numeric_2_1,female_chosen_labels_numeric_3_1,female_chosen_labels_1_1_attack_logical,female_chosen_labels_2_1_attack_logical,female_chosen_labels_3_1_attack_logical,female_chosen_labels_1_1_name,female_chosen_labels_2_1_name,female_chosen_labels_3_1_name,female_chosen_labels_1_1_speaker_id,female_chosen_labels_2_1_speaker_id,female_chosen_labels_3_1_speaker_id,female_chosen_labels_1_1_sex,female_chosen_labels_2_1_sex,female_chosen_labels_3_1_sex  = load_data_female(data_path_female)


import utils.my_functions as my_functions

columns_names,max_name_length = my_functions.get_columns_names_feature_importance(substruct=True)
true_channels_indexes = np.array(my_functions.get_real_channel(np.linspace(start=1, stop=len(columns_names), num=len(columns_names)),len(columns_names)))
true_channels_indexes = true_channels_indexes - 1
true_channels_indexes = true_channels_indexes.astype(int)
columns_names = np.array(columns_names)

female_embedded_groups_1_1 = female_embedded_groups_1_1[:,true_channels_indexes]
female_embedded_groups_1_2 = female_embedded_groups_1_2[:,true_channels_indexes]
female_embedded_groups_1_3 = female_embedded_groups_1_3[:,true_channels_indexes]

male_embedded_groups_1_1 = male_embedded_groups_1_1[:,true_channels_indexes]
male_embedded_groups_1_2 = male_embedded_groups_1_2[:,true_channels_indexes]
male_embedded_groups_1_3 = male_embedded_groups_1_3[:,true_channels_indexes]

from sklearn.preprocessing import StandardScaler
import re

your_list = columns_names[true_channels_indexes]
index_mapping ={}

# Define the custom sorting order for distance metrics
distance_metrics = [
    'Chi-square',
    'Correlation',
    'Hellinger',
    'Intersection',
    'Jensen-Shannon',
    'Kullback-Leibler Divergence',
    'Modified Kolmogorov-Smirnov',
    'Symmetrised Kullback-Leibler'
]

def custom_sort_key(item):
    # Use regex to extract gammatone, gammatone_inv, channel number, and distance metric
    match = re.search(r'filter-(gammatone(?:_inv)?)-channel-(\d+)-distance-(.+?)-\[d_', item)
    if match:
        gammatone = match.group(1)
        channel = int(match.group(2))
        distance_metric = match.group(3)

        # Assign a higher priority to distance_order, a medium priority to channels,
        # and lower priority to 'gammatone' and 'gammatone_inv'
        return (distance_metrics.index(distance_metric), channel, gammatone)

    else:
        # If the regex doesn't match, return the original item to maintain its position
        return (999, 999, "")

# Sort the list based on the custom order
sorted_list = sorted(your_list, key=custom_sort_key)

for new_index, item in enumerate(sorted_list):
    old_index = np.where(columns_names[true_channels_indexes] == item)[0][0]
    index_mapping[old_index] = new_index
    
male_embedded_groups_1_1 = male_embedded_groups_1_1[:,list(index_mapping.keys())]
male_embedded_groups_1_2 = male_embedded_groups_1_2[:,list(index_mapping.keys())]
male_embedded_groups_1_3 = male_embedded_groups_1_3[:,list(index_mapping.keys())]

female_embedded_groups_1_1 = female_embedded_groups_1_1[:,list(index_mapping.keys())]
female_embedded_groups_1_2 = female_embedded_groups_1_2[:,list(index_mapping.keys())]
female_embedded_groups_1_3 = female_embedded_groups_1_3[:,list(index_mapping.keys())]



In [4]:
# define the subchannel model network
import torch
import torch.nn as nn
from utils.AMSloss import AdMSoftmaxLoss
import pickle
# define the subchannel model network
import torch.nn as nn
class SubChannelNetwork(nn.Module):
    def __init__(self, input_channel_size, output_channel_size):
        super(SubChannelNetwork, self).__init__()
        self.input_layer = nn.Linear(input_channel_size, output_channel_size)
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.2)
        self.BN_4 = nn.BatchNorm1d(output_channel_size) 

        
    def forward(self, x):
        x = self.input_layer(x)
        x = self.BN_4(x) 
        x = self.sigmoid(x)
        x = self.dropout(x)
        return x

# define the model network
class DNN(nn.Module):
    def __init__(self, input_channel_size, num_subnetworks, output_channel_size, final_output_size):
        super(DNN, self).__init__()
        self.SubChannelNetwork = nn.ModuleList([
            SubChannelNetwork(input_channel_size, output_channel_size) for _ in range(num_subnetworks)
        ])
        self.fc_between_subnet = nn.Linear(num_subnetworks * output_channel_size,40)
        self.BN = nn.BatchNorm1d(40)
        self.fc = nn.Linear(40, final_output_size)
        self.sigmoid = nn.Sigmoid()
        self.droupout = nn.Dropout(p=0.2)
        self.loss = nn.BCEWithLogitsLoss()
        self.optimizer = None
        self.scheduler = None
        
        
    def forward(self, x):
        subnetwork_outputs = [self.SubChannelNetwork[idx](x[:, idx*input_channel_size:(idx+1)*input_channel_size].to(device)) for idx in range(len(self.SubChannelNetwork))]
        combined_output = torch.cat(subnetwork_outputs, dim=1)
        x = self.fc_between_subnet(combined_output)    
        x = self.BN(x)
        x = self.sigmoid(x)
        x = self.droupout(x)
        output = self.fc(x)
        return output 
    
    @staticmethod
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)



# Just for checking the model and see the number of parameters
num_SubChannelNetwork = 16
input_channel_size = 10
output_channel_size = 5
final_output_size = 1
model = []
model = DNN(input_channel_size, num_SubChannelNetwork, output_channel_size, final_output_size)
model = model.to(device)
print(model)
n = DNN.count_parameters(model)
print("Number of parameters: %s" % n)
model = model.to(device) # send the model to the device

model = pickle.load(open(models_folder + "CM_both_male_9_2.pkl", 'rb'))

model.eval()

spoof_model_male = copy.deepcopy(model)

DNN(
  (SubChannelNetwork): ModuleList(
    (0-15): 16 x SubChannelNetwork(
      (input_layer): Linear(in_features=10, out_features=5, bias=True)
      (sigmoid): Sigmoid()
      (dropout): Dropout(p=0.2, inplace=False)
      (BN_4): BatchNorm1d(5, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (fc_between_subnet): Linear(in_features=80, out_features=40, bias=True)
  (BN): BatchNorm1d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc): Linear(in_features=40, out_features=1, bias=True)
  (sigmoid): Sigmoid()
  (droupout): Dropout(p=0.2, inplace=False)
  (loss): BCEWithLogitsLoss()
)
Number of parameters: 4401


In [5]:
import torch
import torch.nn as nn
from utils.AMSloss import AdMSoftmaxLoss
from utils.OCSoftmaxloss import OCSoftmax
import torch.nn as nn

class SubChannelNetwork(nn.Module):
    def __init__(self, input_channel_size, output_channel_size):
        super(SubChannelNetwork, self).__init__()
        self.input_layer = nn.Linear(input_channel_size, output_channel_size)
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.2)
        self.BN_4 = nn.BatchNorm1d(output_channel_size) 
   
    def forward(self, x):
        res = x
        x = self.input_layer(x)
        x = self.BN_4(x) 
        x = self.sigmoid(x)
        x = self.dropout(x)
        x = res + x 
        return x

class DNN(nn.Module):
    def __init__(self, input_channel_size, num_subnetworks, output_channel_size, final_output_size, r_real , r_fake,alpha):
        super(DNN, self).__init__()
        self.SubChannelNetwork = nn.ModuleList([
            SubChannelNetwork(input_channel_size, output_channel_size) for _ in range(num_subnetworks)
        ])
        self.fc_between_subnet = nn.Linear(num_subnetworks * output_channel_size,40)
        self.BN = nn.BatchNorm1d(40)
        self.fc = nn.Linear(40, final_output_size)
        self.sigmoid = nn.Sigmoid()
        self.droupout = nn.Dropout(p=0.2)
        self.r_real = r_real
        self.r_fake = r_fake
        self.alpha = alpha
        self.loss = OCSoftmax(feat_dim = final_output_size, r_real = self.r_real, r_fake = self.r_fake, alpha = self.alpha)
        self.optimizer = None
        self.scheduler = None
        
        
    def forward(self, x):
        subnetwork_outputs = [self.SubChannelNetwork[idx](x[:, idx*input_channel_size:(idx+1)*input_channel_size].to(device)) for idx in range(len(self.SubChannelNetwork))]
        combined_output = torch.cat(subnetwork_outputs, dim=1)
        x = self.fc_between_subnet(combined_output)    
        x = self.BN(x)
        x = self.sigmoid(x)
        x = self.droupout(x)
        output = self.fc(x)
        return output 
    
    @staticmethod
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

r_real = 0.5 
r_fake = 0.1
alpha = 20
# Example usage
num_SubChannelNetwork = 16
input_channel_size = 10
output_channel_size = 10
final_output_size = 16*3
model = DNN(input_channel_size, num_SubChannelNetwork, output_channel_size, final_output_size,r_real = r_real , r_fake = r_fake,alpha = alpha)


model = model.to(device) # send the model to the device

model = pickle.load(open(models_folder + "CM_both_female_10_17.pkl", 'rb'))

model.eval()

spoof_model_female = copy.deepcopy(model)


In [6]:
scaler_male = StandardScaler(with_mean = True, with_std = True)
scaler_male.fit(male_embedded_groups_1_1)
mean_features = scaler_male.mean_
std_features = scaler_male.scale_
male_embedded_groups_1_1_norm = scaler_male.transform(male_embedded_groups_1_1)
male_embedded_groups_1_2_norm = scaler_male.transform(male_embedded_groups_1_2)
male_embedded_groups_1_3_norm = scaler_male.transform(male_embedded_groups_1_3)


scaler_female = StandardScaler(with_mean = True, with_std = True)
scaler_female.fit(female_embedded_groups_1_1)
mean_features = scaler_female.mean_
std_features = scaler_female.scale_
female_embedded_groups_1_1_norm = scaler_female.transform(female_embedded_groups_1_1)
female_embedded_groups_1_2_norm = scaler_female.transform(female_embedded_groups_1_2)
female_embedded_groups_1_3_norm = scaler_female.transform(female_embedded_groups_1_3)

In [7]:
embedded_groups_1_1_norm,embedded_groups_1_2_norm,embedded_groups_1_3_norm,chosen_labels_1_1_is_spoofed,chosen_labels_2_1_is_spoofed,chosen_labels_3_1_is_spoofed,chosen_labels_numeric_1_1,chosen_labels_numeric_2_1,chosen_labels_numeric_3_1,chosen_labels_1_1_attack_logical,chosen_labels_2_1_attack_logical,chosen_labels_3_1_attack_logical,chosen_labels_1_1_name,chosen_labels_2_1_name,chosen_labels_3_1_name,chosen_labels_1_1_speaker_id,chosen_labels_2_1_speaker_id,chosen_labels_3_1_speaker_id,chosen_labels_1_1_sex,chosen_labels_2_1_sex,chosen_labels_3_1_sex    =  concatenate_data(male_embedded_groups_1_1_norm,male_embedded_groups_1_2_norm,male_embedded_groups_1_3_norm,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_1_1_is_spoofed,male_chosen_labels_2_1_is_spoofed,male_chosen_labels_3_1_is_spoofed,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_numeric_1_1,male_chosen_labels_numeric_2_1,male_chosen_labels_numeric_3_1,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_1_1_attack_logical,male_chosen_labels_2_1_attack_logical,male_chosen_labels_3_1_attack_logical,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_1_1_name,male_chosen_labels_2_1_name,male_chosen_labels_3_1_name,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_1_1_speaker_id,male_chosen_labels_2_1_speaker_id, male_chosen_labels_3_1_speaker_id,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    male_chosen_labels_1_1_sex,male_chosen_labels_2_1_sex,male_chosen_labels_3_1_sex,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_embedded_groups_1_1_norm,female_embedded_groups_1_2_norm,female_embedded_groups_1_3_norm,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_1_1_is_spoofed,female_chosen_labels_2_1_is_spoofed,female_chosen_labels_3_1_is_spoofed,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_numeric_1_1,female_chosen_labels_numeric_2_1,female_chosen_labels_numeric_3_1,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_1_1_attack_logical,female_chosen_labels_2_1_attack_logical,female_chosen_labels_3_1_attack_logical,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_1_1_name,female_chosen_labels_2_1_name,female_chosen_labels_3_1_name,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_1_1_speaker_id,female_chosen_labels_2_1_speaker_id,female_chosen_labels_3_1_speaker_id,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    female_chosen_labels_1_1_sex,female_chosen_labels_2_1_sex,female_chosen_labels_3_1_sex)


In [8]:
embedded_groups_1_1,embedded_groups_1_2,embedded_groups_1_3,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_    =  concatenate_data(male_embedded_groups_1_1,male_embedded_groups_1_2,male_embedded_groups_1_3,
                                                                                                                        male_chosen_labels_1_1_is_spoofed,male_chosen_labels_2_1_is_spoofed,male_chosen_labels_3_1_is_spoofed,
                                                                                                                        male_chosen_labels_numeric_1_1,male_chosen_labels_numeric_2_1,male_chosen_labels_numeric_3_1,
                                                                                                                        male_chosen_labels_1_1_attack_logical,male_chosen_labels_2_1_attack_logical,male_chosen_labels_3_1_attack_logical,
                                                                                                                        male_chosen_labels_1_1_name,male_chosen_labels_2_1_name,male_chosen_labels_3_1_name,
                                                                                                                        male_chosen_labels_1_1_speaker_id,male_chosen_labels_2_1_speaker_id, male_chosen_labels_3_1_speaker_id,
                                                                                                                        male_chosen_labels_1_1_sex,male_chosen_labels_2_1_sex,male_chosen_labels_3_1_sex,
                                                                                                                        
                                                                                                                        female_embedded_groups_1_1,female_embedded_groups_1_2,female_embedded_groups_1_3,
                                                                                                                        female_chosen_labels_1_1_is_spoofed,female_chosen_labels_2_1_is_spoofed,female_chosen_labels_3_1_is_spoofed,
                                                                                                                        female_chosen_labels_numeric_1_1,female_chosen_labels_numeric_2_1,female_chosen_labels_numeric_3_1,
                                                                                                                        female_chosen_labels_1_1_attack_logical,female_chosen_labels_2_1_attack_logical,female_chosen_labels_3_1_attack_logical,
                                                                                                                        female_chosen_labels_1_1_name,female_chosen_labels_2_1_name,female_chosen_labels_3_1_name,
                                                                                                                        female_chosen_labels_1_1_speaker_id,female_chosen_labels_2_1_speaker_id,female_chosen_labels_3_1_speaker_id,
                                                                                                                        female_chosen_labels_1_1_sex,female_chosen_labels_2_1_sex,female_chosen_labels_3_1_sex)
scaler_all = StandardScaler(with_mean = True, with_std = True)
scaler_all.fit(embedded_groups_1_1)
mean_features = scaler_female.mean_
std_features = scaler_female.scale_
embedded_groups_1_1_all = scaler_all.transform(embedded_groups_1_1)
embedded_groups_1_2_all = scaler_all.transform(embedded_groups_1_2)
embedded_groups_1_3_all = scaler_all.transform(embedded_groups_1_3)

In [9]:
train_dataset_all = dataset(data = embedded_groups_1_1_norm , is_spoofed = chosen_labels_1_1_is_spoofed , chosen_labels_numeric = chosen_labels_numeric_1_1,
                        attack_logical = chosen_labels_1_1_attack_logical, name = chosen_labels_1_1_name , speaker_id = chosen_labels_1_1_speaker_id, sex = chosen_labels_1_1_sex ,data_transform = None , labels_transform = None,
                        data_for_gender_classification = None,data_without_separation = embedded_groups_1_1_all);

Dev_dataset_all = dataset(data = embedded_groups_1_2_norm , is_spoofed = chosen_labels_2_1_is_spoofed , chosen_labels_numeric = chosen_labels_numeric_2_1,
                        attack_logical = chosen_labels_2_1_attack_logical, name = chosen_labels_2_1_name , speaker_id = chosen_labels_2_1_speaker_id, sex = chosen_labels_2_1_sex ,  data_transform = None , labels_transform = None,
                        data_for_gender_classification = None,data_without_separation = embedded_groups_1_2_all);


Eval_dataset_all = dataset(data = embedded_groups_1_3_norm , is_spoofed = chosen_labels_3_1_is_spoofed , chosen_labels_numeric = chosen_labels_numeric_3_1,
                        attack_logical = chosen_labels_3_1_attack_logical, name = chosen_labels_3_1_name , speaker_id = chosen_labels_3_1_speaker_id, sex = chosen_labels_3_1_sex , data_transform = None , labels_transform = None,
                        data_for_gender_classification = None,data_without_separation = embedded_groups_1_3_all);


In [10]:
from ASV_utils.data_loading import *


models_folder = "ECAPA_TDNN/inference_models/models_both_not_normalize/"

data_path_male = "Data/male_vs_female_DB_models/16_bits/none/male/"

data_path_female = "Data/male_vs_female_DB_models/16_bits/none/female/"

g_male_embedded_groups_1_1,g_male_embedded_groups_1_2,g_male_embedded_groups_1_3,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_  = load_data_male(data_path_male)

g_female_embedded_groups_1_1,g_female_embedded_groups_1_2,g_female_embedded_groups_1_3,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_  = load_data_female(data_path_female)

embedded_groups_1_1,embedded_groups_1_2,embedded_groups_1_3,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_    =  concatenate_data(g_male_embedded_groups_1_1,g_male_embedded_groups_1_2,g_male_embedded_groups_1_3,
                                                                                                                    male_chosen_labels_1_1_is_spoofed,male_chosen_labels_2_1_is_spoofed,male_chosen_labels_3_1_is_spoofed,
                                                                                                                    male_chosen_labels_numeric_1_1,male_chosen_labels_numeric_2_1,male_chosen_labels_numeric_3_1,
                                                                                                                    male_chosen_labels_1_1_attack_logical,male_chosen_labels_2_1_attack_logical,male_chosen_labels_3_1_attack_logical,
                                                                                                                    male_chosen_labels_1_1_name,male_chosen_labels_2_1_name,male_chosen_labels_3_1_name,
                                                                                                                    male_chosen_labels_1_1_speaker_id,male_chosen_labels_2_1_speaker_id, male_chosen_labels_3_1_speaker_id,
                                                                                                                    male_chosen_labels_1_1_sex,male_chosen_labels_2_1_sex,male_chosen_labels_3_1_sex,
                                                                                                                    
                                                                                                                    g_female_embedded_groups_1_1,g_female_embedded_groups_1_2,g_female_embedded_groups_1_3,
                                                                                                                    female_chosen_labels_1_1_is_spoofed,female_chosen_labels_2_1_is_spoofed,female_chosen_labels_3_1_is_spoofed,
                                                                                                                    female_chosen_labels_numeric_1_1,female_chosen_labels_numeric_2_1,female_chosen_labels_numeric_3_1,
                                                                                                                    female_chosen_labels_1_1_attack_logical,female_chosen_labels_2_1_attack_logical,female_chosen_labels_3_1_attack_logical,
                                                                                                                    female_chosen_labels_1_1_name,female_chosen_labels_2_1_name,female_chosen_labels_3_1_name,
                                                                                                                    female_chosen_labels_1_1_speaker_id,female_chosen_labels_2_1_speaker_id,female_chosen_labels_3_1_speaker_id,
                                                                                                                    female_chosen_labels_1_1_sex,female_chosen_labels_2_1_sex,female_chosen_labels_3_1_sex)
chosen_labels_1_1_sex = np.array([elem[0] for elem in chosen_labels_1_1_sex])
                                 
chosen_labels_2_1_sex = np.array([elem[0] for elem in chosen_labels_2_1_sex])

chosen_labels_3_1_sex = np.array([elem[0] for elem in chosen_labels_3_1_sex])


In [11]:



import utils.my_functions as my_functions

columns_names,max_name_length = my_functions.get_columns_names_feature_importance(substruct=True)
true_channels_indexes = np.array(my_functions.get_real_channel(np.linspace(start=1, stop=len(columns_names), num=len(columns_names)),len(columns_names)))
true_channels_indexes = true_channels_indexes - 1
true_channels_indexes = true_channels_indexes.astype(int)
columns_names = np.array(columns_names)
'''
g_female_embedded_groups_1_1 = g_female_embedded_groups_1_1[:,true_channels_indexes]
g_female_embedded_groups_1_2 = g_female_embedded_groups_1_2[:,true_channels_indexes]
g_female_embedded_groups_1_3 = g_female_embedded_groups_1_3[:,true_channels_indexes]

g_male_embedded_groups_1_1 = g_male_embedded_groups_1_1[:,true_channels_indexes]
g_male_embedded_groups_1_2 = g_male_embedded_groups_1_2[:,true_channels_indexes]
g_male_embedded_groups_1_3 = g_male_embedded_groups_1_3[:,true_channels_indexes]
'''

embedded_groups_1_1 = embedded_groups_1_1[:,true_channels_indexes]
embedded_groups_1_2 = embedded_groups_1_2[:,true_channels_indexes]
embedded_groups_1_3 = embedded_groups_1_3[:,true_channels_indexes]

from sklearn.preprocessing import StandardScaler
import re

your_list = columns_names[true_channels_indexes]
index_mapping ={}

# Define the custom sorting order for distance metrics
distance_metrics = [
    'Chi-square',
    'Correlation',
    'Hellinger',
    'Intersection',
    'Jensen-Shannon',
    'Kullback-Leibler Divergence',
    'Modified Kolmogorov-Smirnov',
    'Symmetrised Kullback-Leibler'
]

def custom_sort_key(item):
    # Use regex to extract gammatone, gammatone_inv, channel number, and distance metric
    match = re.search(r'filter-(gammatone(?:_inv)?)-channel-(\d+)-distance-(.+?)-\[d_', item)
    if match:
        gammatone = match.group(1)
        channel = int(match.group(2))
        distance_metric = match.group(3)

        # Assign a higher priority to distance_order, a medium priority to channels,
        # and lower priority to 'gammatone' and 'gammatone_inv'
        return (distance_metrics.index(distance_metric), channel, gammatone)

    else:
        # If the regex doesn't match, return the original item to maintain its position
        return (999, 999, "")

# Sort the list based on the custom order
sorted_list = sorted(your_list, key=custom_sort_key)

for new_index, item in enumerate(sorted_list):
    old_index = np.where(columns_names[true_channels_indexes] == item)[0][0]
    index_mapping[old_index] = new_index
    

embedded_groups_1_1 = embedded_groups_1_1[:,list(index_mapping.keys())]
embedded_groups_1_2 = embedded_groups_1_2[:,list(index_mapping.keys())]
embedded_groups_1_3 = embedded_groups_1_3[:,list(index_mapping.keys())]




In [12]:
scaler = StandardScaler(with_mean = True, with_std = True)
scaler.fit(embedded_groups_1_1)
mean_features = scaler.mean_
std_features = scaler.scale_

embedded_groups_1_1 = scaler.transform(embedded_groups_1_1)
embedded_groups_1_2 = scaler.transform(embedded_groups_1_2)
embedded_groups_1_3 = scaler.transform(embedded_groups_1_3)

train_dataset_all.data_for_gender_classification = embedded_groups_1_1
Dev_dataset_all.data_for_gender_classification   = embedded_groups_1_2
Eval_dataset_all.data_for_gender_classification  = embedded_groups_1_3


train_dataset_all.sex = pd.Series([elem[0] for elem in train_dataset_all.sex])
Dev_dataset_all.sex = pd.Series([elem[0] for elem in Dev_dataset_all.sex])
Eval_dataset_all.sex = pd.Series([elem[0] for elem in Eval_dataset_all.sex])

train_dataset_all.name = np.array([str(elem).split("/")[-1].split("'")[0].split('.flac')[0] for elem in train_dataset_all.name])

Dev_dataset_all.name = np.array([str(elem).split("/")[-1].split("'")[0].split('.flac')[0] for elem in Dev_dataset_all.name])

Eval_dataset_all.name = np.array([str(elem).split("/")[-1].split("'")[0].split('.flac')[0] for elem in Eval_dataset_all.name])

In [13]:
labels_eval_female = torch.Tensor(Eval_dataset_all.is_spoofed[(Eval_dataset_all.sex == 'female').values].values).cpu().numpy().copy()
total_data = torch.Tensor(Eval_dataset_all.data[(Eval_dataset_all.sex == 'female').values]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    output = spoof_model_female(torch.Tensor(total_data).to(device))
    _ , score_eval_female = spoof_model_female.loss(torch.Tensor(output).to(device),None)
    score_eval_female = -1*score_eval_female
    
score_eval_female = score_eval_female.cpu().numpy().copy()

eer_eval_female, thr = my_functions.compute_eer(labels_eval_female,score_eval_female) # compute equal error rate

print(f"\tEval Female labels gender EER: ({eer_eval_female}%), {thr}")


	Eval Female labels gender EER: (0.10177271288382395%), -0.8237718060240031


In [14]:
labels_eval_male = torch.Tensor(Eval_dataset_all.is_spoofed[(Eval_dataset_all.sex == 'male').values].values).cpu().numpy().copy()
total_data = torch.Tensor(Eval_dataset_all.data[(Eval_dataset_all.sex == 'male').values]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    score_eval_male = torch.sigmoid(spoof_model_male(torch.Tensor(total_data).to(device)))
    
score_eval_male = score_eval_male.cpu().numpy().copy()

eer_eval_male, thr = my_functions.compute_eer(labels_eval_male,score_eval_male) # compute equal error rate

print(f"\t Eval Male EER: ({100*eer_eval_male}%) , {thr}")

	 Eval Male EER: (9.203296703296708%) , 0.04938841153161375


# implementation of t-DCF in 2 subsystems

In [15]:
bonafide_score_cm_female_eval = score_eval_female[labels_eval_female == 0]
bonafide_score_cm_female_eval = -1*bonafide_score_cm_female_eval

spoof_score_cm_female_eval = score_eval_female[labels_eval_female == 1]
spoof_score_cm_female_eval  = -1*spoof_score_cm_female_eval

Prior_spoof = 0.05

target_scores_female_eval = results_eval.loc[(results_eval['asv_label'] ==  "target") & (results_eval['its_male_ground_truth'] == 'female') ]["asv_score_female"].values
nontarget_scores_female_eval = results_eval.loc[(results_eval['asv_label'] == "nontarget") & (results_eval['its_male_ground_truth'] == 'female') ]["asv_score_female"].values
spoof_scores_female_eval = results_eval.loc[(results_eval['asv_label'] ==  "spoof") & (results_eval['its_male_ground_truth'] == 'female')]["asv_score_female"].values

list_asv_score_female = list_asv_score = [0.401]
type = 'constrained'
_ ,_,_  = compute_t_dcf(bonafide_score_cm_female_eval,spoof_score_cm_female_eval,Prior_spoof,target_scores_female_eval,nontarget_scores_female_eval,spoof_scores_female_eval,list_asv_score,type)

The t-DCF evaluation type is: constrained
t-DCF evaluation from [Nbona=5072, Nspoof=44226] trials

t-DCF MODEL
   Ptar         =  0.94050 (Prior probability of target user)
   Pnon         =  0.00950 (Prior probability of nontarget user)
   Pspoof       =  0.05000 (Prior probability of spoofing attack)
   Cfa_asv      = 10.00000 (Cost of ASV falsely accepting a nontarget)
   Cmiss_asv    =  1.00000 (Cost of ASV falsely rejecting target speaker)
   Cfa_cm       = 10.00000 (Cost of CM falsely passing a spoof to ASV system)
   Cmiss_cm     =  1.00000 (Cost of CM falsely blocking target utterance which never reaches ASV)

   Implied normalized t-DCF function (depends on t-DCF parameters and ASV errors), s=CM threshold)
   tDCF_norm(s) =  3.00467 x Pmiss_cm(s) + Pfa_cm(s)

the asv threshold is from eer on asv development set  0.401
the CM thresholds is:  [-0.83954342 -0.83854342 -0.83505869 ...  0.95363593  0.95442975
  0.95455223]
the CM threshold min is: 0.7075192928314209
the tDCF_norm i

In [16]:

bonafide_score_cm_male_eval = 1-score_eval_male[labels_eval_male == 0].flatten()
spoof_score_cm_male_eval = 1-score_eval_male[labels_eval_male == 1].flatten()

Prior_spoof = 0.05

target_scores_male_eval = results_eval.loc[(results_eval['asv_label'] ==  "target") & (results_eval['its_male_ground_truth'] == 'male') ]["asv_score_male"].values
nontarget_scores_male_eval = results_eval.loc[(results_eval['asv_label'] == "nontarget") & (results_eval['its_male_ground_truth'] == 'male') ]["asv_score_male"].values
spoof_scores_male_eval = results_eval.loc[(results_eval['asv_label'] ==  "spoof") & (results_eval['its_male_ground_truth'] == 'male')]["asv_score_male"].values

list_asv_score_male = list_asv_score = [0.352]
type = 'constrained'
_ ,_,_  = compute_t_dcf(bonafide_score_cm_male_eval,spoof_score_cm_male_eval,Prior_spoof,target_scores_male_eval,nontarget_scores_male_eval,spoof_scores_male_eval,list_asv_score,type)

The t-DCF evaluation type is: constrained
t-DCF evaluation from [Nbona=2283, Nspoof=19656] trials

t-DCF MODEL
   Ptar         =  0.94050 (Prior probability of target user)
   Pnon         =  0.00950 (Prior probability of nontarget user)
   Pspoof       =  0.05000 (Prior probability of spoofing attack)
   Cfa_asv      = 10.00000 (Cost of ASV falsely accepting a nontarget)
   Cmiss_asv    =  1.00000 (Cost of ASV falsely rejecting target speaker)
   Cfa_cm       = 10.00000 (Cost of CM falsely passing a spoof to ASV system)
   Cmiss_cm     =  1.00000 (Cost of CM falsely blocking target utterance which never reaches ASV)

   Implied normalized t-DCF function (depends on t-DCF parameters and ASV errors), s=CM threshold)
   tDCF_norm(s) =  2.72851 x Pmiss_cm(s) + Pfa_cm(s)

the asv threshold is from eer on asv development set  0.352
the CM thresholds is:  [3.27395439e-04 1.32739544e-03 1.33728981e-03 ... 9.96948719e-01
 9.97036994e-01 9.97085333e-01]
the CM threshold min is: 0.87405800819396

In [17]:

def compute_a_det_curve(target_scores_male,target_scores_female, nontarget_scores_male,nontarget_scores_female, spoof_scores_male,spoof_scores_female):

    trg_scores = np.concatenate((target_scores_male, target_scores_female))
    nontrg_scores = np.concatenate((nontarget_scores_male, nontarget_scores_female))
    spf_scores = np.concatenate((spoof_scores_male,spoof_scores_female))
    
    all_scores = np.concatenate((trg_scores, nontrg_scores, spf_scores))
    labels = np.concatenate(
        (np.ones_like(trg_scores), np.zeros_like(nontrg_scores), np.ones_like(spf_scores) + 1))

    # Sort labels based on scores
    indices = np.argsort(all_scores, kind='mergesort')
    labels = labels[indices]
    scores_sorted = all_scores[indices]

    fp_nontrg, fp_spf, fn = len(nontrg_scores), len(spf_scores), 0
    far_asvs, far_cms, frrs, a_dcf_thresh = [1.], [1.], [0.], [float(np.min(scores_sorted))-1e-8]
    for sco, lab in zip(scores_sorted, labels):
        if lab == 0: # non-target
            fp_nontrg -= 1 # false alarm for accepting nontarget trial
        elif lab == 1: # target
            fn += 1 # miss
        elif lab == 2: # spoof
            fp_spf -= 1 # false alarm for accepting spof trial
        else:
            raise ValueError ("Label should be one of (0, 1, 2).")
        far_asvs.append(fp_nontrg / len(nontrg_scores))
        far_cms.append(fp_spf / len(spf_scores))
        frrs.append(fn / len(trg_scores))
        a_dcf_thresh.append(sco)

    return far_asvs, far_cms, frrs, a_dcf_thresh

In [18]:
def calc_scores(cm_filenames, cm_scores, asv_speakers,asv_score_male,asv_score_female,asv_labels, asv_male_filenames,asv_female_filenames,cm_speakers_sex, thr_male,thr_female):   

    # Create a dictionary to map filenames to cm_scores
    
    asv_filenames = np.concatenate((asv_male_filenames,asv_female_filenames))
    
    cm_score_dict = dict(zip(cm_filenames, cm_scores))
    cm_speakers_sex_dict = dict(zip(cm_filenames,cm_speakers_sex))
    asv_labels_dict = dict(zip(asv_filenames, asv_labels))
    asv_score_dict_male = dict(zip(asv_male_filenames, asv_score_male))
    asv_score_dict_female = dict(zip(asv_female_filenames, asv_score_female))
    asv_speakers_dict = dict(zip(asv_filenames, asv_speakers))
    
    
   
    # Iterate over the thresholds
    adjusted_scores = []
    adjusted_labels = []
    adjusted_filename = []
    adjusted_speakers = []  
    
    # Iterate over the asv_filenames and adjust cm_scores based on threshold
    for filename in asv_filenames:
        if filename in cm_score_dict:
            cm_score = cm_score_dict[filename]
            cm_sex = cm_speakers_sex_dict[filename]
            if cm_sex == 'male' or cm_sex == 1:
                if cm_score >= thr_male: # If the score is greater than the threshold, append the score - bonafide
                    if filename in asv_score_dict_male:
                        adjusted_scores.append(asv_score_dict_male[filename])
                    elif filename in asv_score_dict_female:
                        adjusted_scores.append(asv_score_dict_female[filename])
                    else:
                        raise ValueError("Filename not found")
                    adjusted_labels.append(asv_labels_dict[filename])
                    adjusted_speakers.append(asv_speakers_dict[filename])
                    adjusted_filename.append(adjusted_filename)  
                    
                else:
                    adjusted_scores.append(-np.inf) # If the score is lower than the threshold, append the score - spoof
                    adjusted_labels.append(asv_labels_dict[filename])
                    adjusted_speakers.append(asv_speakers_dict[filename])
                    adjusted_filename.append(adjusted_filename)  
            elif cm_sex == 'female' or cm_sex == 0:
                if cm_score >= thr_female: # If the score is greater than the threshold, append the score - bonafide
                    
                    if filename in asv_score_dict_female:
                        adjusted_scores.append(asv_score_dict_female[filename])
                    elif filename in asv_score_dict_male:
                        adjusted_scores.append(asv_score_dict_male[filename])
                    else:
                        raise ValueError("Filename not found")
                    adjusted_labels.append(asv_labels_dict[filename])
                    adjusted_speakers.append(asv_speakers_dict[filename])
                    adjusted_filename.append(adjusted_filename)  
                    
                else:
                    adjusted_scores.append(-np.inf) # If the score is lower than the threshold, append the score - spoof
                    adjusted_labels.append(asv_labels_dict[filename])
                    adjusted_speakers.append(asv_speakers_dict[filename])
                    adjusted_filename.append(adjusted_filename)  
        else:
            # If the filename is not found, append -np.inf or handle accordingly
            print("Error! Filename {filename} not found in cm_score_dict")
            
    return adjusted_scores, adjusted_labels,adjusted_speakers,adjusted_filename
                    

In [19]:
from tqdm import tqdm
#using only one CM threshold
import multiprocessing
import a_DCF.a_DCF_package.a_dcf.a_dcf as a_dcf
print_cost = True

class CostModel:
    "Class describing SASV-DCF's relevant costs" #according to ASVSpoof 5
    Pspf: float = 0.05
    Pnontrg: float = 0.0095
    Ptrg: float = 0.9405
    Cmiss: float = 1
    Cfa_asv: float = 10
    Cfa_cm: float = 10


def calc_a_dcf_spesific(cm_filenames,cm_speakers_sex,asv_male_filenames,asv_female_filenames,asv_speakers,asv_labels,score_cm_male,asv_score_male,cm_thr_male,score_cm_female,asv_score_female,cm_thr_female):
    '''
    cm filenames: list of filenames for the CM system
    cm_speakers_sex: list of speakers for the CM system
    asv_filenames: list of filenames for the ASV system
    asv_speakers: list of speakers for the ASV system
    asv_labels: list of labels for the ASV system
    score_cm_male: list of male scores for the CM system
    asv_score_male: list of male scores for the ASV system
    cm_thr_male: threshold for Male CM system
    score_cm_female: list of female scores for the CM system
    asv_score_female: list of female scores for the ASV system
    cm_thr_female: threshold for Female CM system
    
    '''

    cost_model = CostModel()
    
    
    cm_scores = np.concatenate((score_cm_male,score_cm_female))
    
    #need to think about that how to send the scores
    
    # what input to the sasv_dcf function over all the threshold and if not input -inf
    adjusted_scores, adjusted_labels,_,_ = calc_scores(cm_filenames, cm_scores, asv_speakers,asv_score_male,asv_score_female,asv_labels, asv_male_filenames,asv_female_filenames,cm_speakers_sex, cm_thr_male,cm_thr_female)


    # input the scores to the a-dcf function
    
    min_a_dcf , min_a_dcf_thresh,a_dcfs_normed,a_dcf_thresh = a_dcf.calculate_a_dcf_python(asv_scores = np.array(adjusted_scores), asv_labels = np.array(adjusted_labels),cost_model = cost_model, printres = True)

    
            
    print(f"Male CM Threshold: {cm_thr_male},Female CM Threshold: {cm_thr_female}, \tMin a-dcf: {min_a_dcf} \tMin a-dcf threshold: {min_a_dcf_thresh}")
    
    
    return min_a_dcf , min_a_dcf_thresh,a_dcfs_normed,a_dcf_thresh

# Eval with Labels:

In [20]:
import pickle

#asv male

# cm male
score_cm_male = 1-score_eval_male.flatten()
cm_thr_male =   0.76 # validation thr

#cm female
score_cm_female = -1*score_eval_female.flatten()
cm_thr_female =   0.5100000000000013 # validation thr

cm_filenames = np.concatenate((Eval_dataset_all.name[Eval_dataset_all.sex == 'male'], Eval_dataset_all.name[Eval_dataset_all.sex == 'female']))

cm_speakers_sex = np.concatenate((Eval_dataset_all.sex[Eval_dataset_all.sex == 'male'].values, Eval_dataset_all.sex[Eval_dataset_all.sex == 'female'].values))

asv_male_filenames = results_eval.loc[(results_eval['its_male_ground_truth'] == 'male')]["file_name"].values

asv_female_filenames = results_eval.loc[(results_eval['its_male_ground_truth'] == 'female')]["file_name"].values

asv_score_male = results_eval.loc[(results_eval['its_male_ground_truth'] == 'male')]["asv_score_male"].values

asv_score_female = results_eval.loc[(results_eval['its_male_ground_truth'] == 'female')]["asv_score_female"].values

asv_speakers =  np.concatenate((results_eval.loc[(results_eval['its_male_ground_truth'] == 'male')]["speaker_id_male"].values,results_eval.loc[(results_eval['its_male_ground_truth'] == 'female')]["speaker_id_male"].values))

asv_labels = np.concatenate((results_eval.loc[(results_eval['its_male_ground_truth'] == 'male')]["asv_label"].values,results_eval.loc[(results_eval['its_male_ground_truth'] == 'female')]["asv_label"].values))

calc_a_dcf_spesific(cm_filenames,cm_speakers_sex,asv_male_filenames,asv_female_filenames,asv_speakers,asv_labels,score_cm_male,asv_score_male,cm_thr_male,score_cm_female,asv_score_female,cm_thr_female)


a-DCF: 0.16633, threshold: 0.36817
Male CM Threshold: 0.76,Female CM Threshold: 0.5100000000000013, 	Min a-dcf: 0.16632516168217715 	Min a-dcf threshold: 0.36816923726688733


(0.16632516168217715,
 0.36816923726688733,
 array([1.        , 1.00029435, 1.0005887 , ..., 1.58008356, 1.58037792,
        1.58067227]),
 array([      -inf,       -inf,       -inf, ..., 0.83300618, 0.83519231,
        0.83914347]))

In [21]:
import pickle

#asv male

# cm male
score_cm_male = 1-score_eval_male.flatten()
cm_thr_male =  0.88 # from the plot

#cm female
score_cm_female = -1*score_eval_female.flatten()
cm_thr_female =  0.7000000000000015 # from the plot

cm_filenames = np.concatenate((Eval_dataset_all.name[Eval_dataset_all.sex == 'male'], Eval_dataset_all.name[Eval_dataset_all.sex == 'female']))

cm_speakers_sex = np.concatenate((Eval_dataset_all.sex[Eval_dataset_all.sex == 'male'].values, Eval_dataset_all.sex[Eval_dataset_all.sex == 'female'].values))

asv_male_filenames = results_eval.loc[(results_eval['its_male_ground_truth'] == 'male')]["file_name"].values

asv_female_filenames = results_eval.loc[(results_eval['its_male_ground_truth'] == 'female')]["file_name"].values

asv_score_male = results_eval.loc[(results_eval['its_male_ground_truth'] == 'male')]["asv_score_male"].values

asv_score_female = results_eval.loc[(results_eval['its_male_ground_truth'] == 'female')]["asv_score_female"].values

asv_speakers =  np.concatenate((results_eval.loc[(results_eval['its_male_ground_truth'] == 'male')]["speaker_id_male"].values,results_eval.loc[(results_eval['its_male_ground_truth'] == 'female')]["speaker_id_female"].values))

asv_labels = np.concatenate((results_eval.loc[(results_eval['its_male_ground_truth'] == 'male')]["asv_label"].values,results_eval.loc[(results_eval['its_male_ground_truth'] == 'female')]["asv_label"].values))

calc_a_dcf_spesific(cm_filenames,cm_speakers_sex,asv_male_filenames,asv_female_filenames,asv_speakers,asv_labels,score_cm_male,asv_score_male,cm_thr_male,score_cm_female,asv_score_female,cm_thr_female)


a-DCF: 0.11721, threshold: 0.36817
Male CM Threshold: 0.88,Female CM Threshold: 0.7000000000000015, 	Min a-dcf: 0.11721403382600303 	Min a-dcf threshold: 0.36816923726688733


(0.11721403382600303,
 0.36816923726688733,
 array([1.        , 1.00029435, 1.0005887 , ..., 1.58008356, 1.58037792,
        1.58067227]),
 array([      -inf,       -inf,       -inf, ..., 0.83300618, 0.83519231,
        0.83914347]))

# Eval with Preds:

In [22]:
import torchaudio
from speechbrain.pretrained import EncoderClassifier

# load the model - the moodel that check if it's female or male
with open(os.path.join(models_folder,"gender_XGB_model_both_norm_male_vs_female_db_models.pkl"), 'rb') as fp:
#with open(os.path.join(models_folder,"gender_XGB_model_both_norm_male_vs_female_db_models.pkl"), 'rb') as fp:
    gender_model = pickle.load(fp)
    


In [23]:
labels_eval_male_gender_classification = torch.Tensor(Eval_dataset_all.is_spoofed[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 1]).cpu().numpy().copy()
total_data = torch.Tensor(Eval_dataset_all.data[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 1]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    score_eval_male_gender_classification = torch.sigmoid(spoof_model_male(torch.Tensor(total_data).to(device)))
    
score_eval_male_gender_classification = score_eval_male_gender_classification.cpu().numpy().copy()

eval_male_eer, _ = my_functions.compute_eer(labels_eval_male_gender_classification,score_eval_male_gender_classification) # compute equal error rate

print(f"\tEval male with gender classification EER: ({100*eval_male_eer}%) ")

	Eval male with gender classification EER: (9.069893266440404%) 


In [24]:
labels_eval_male_gender_classification

array([1., 0., 1., ..., 1., 1., 1.], dtype=float32)

In [25]:
labels_eval_female_gender_classification = torch.Tensor(Eval_dataset_all.is_spoofed[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 0].values).cpu().numpy().copy()
total_data = torch.Tensor(Eval_dataset_all.data[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 0]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    output = spoof_model_female(torch.Tensor(total_data).to(device))
    _ , score_eval_female_gender_classification = spoof_model_female.loss(torch.Tensor(output).to(device),None)
    score_eval_female_gender_classification = -1*score_eval_female_gender_classification
    
score_eval_female_gender_classification = score_eval_female_gender_classification.cpu().numpy().copy()

eer, _ = my_functions.compute_eer(labels_eval_female_gender_classification,score_eval_female_gender_classification) # compute equal error rate

print(f"\tDev Female EER:({100*eer}%)")

	Dev Female EER:(10.318936419363473%)


In [26]:
bonafide_score_cm_female_eval = score_eval_female_gender_classification[labels_eval_female_gender_classification == 0]
bonafide_score_cm_female_eval = -1*bonafide_score_cm_female_eval

spoof_score_cm_female_eval = score_eval_female_gender_classification[labels_eval_female_gender_classification == 1]
spoof_score_cm_female_eval  = -1*spoof_score_cm_female_eval


target_scores_female_eval = results_eval.loc[(results_eval['asv_label'] ==  "target") & (results_eval['its_male'] == False) ]["asv_score_female"].values
nontarget_scores_female_eval = results_eval.loc[(results_eval['asv_label'] == "nontarget") & (results_eval['its_male'] == False) ]["asv_score_female"].values
spoof_scores_female_eval = results_eval.loc[(results_eval['asv_label'] ==  "spoof") & (results_eval['its_male'] == False)]["asv_score_female"].values
list_asv_score_female = list_asv_score = [0.401]


bonafide_score_cm_male_eval = score_eval_male_gender_classification[labels_eval_male_gender_classification == 0].flatten()
bonafide_score_cm_male_eval = 1-bonafide_score_cm_male_eval

spoof_score_cm_male_eval = score_eval_male_gender_classification[labels_eval_male_gender_classification == 1].flatten()
spoof_score_cm_male_eval = 1-spoof_score_cm_male_eval

Prior_spoof = 0.05
target_scores_male_eval = results_eval.loc[(results_eval['asv_label'] ==  "target") & (results_eval['its_male'] == True) ]["asv_score_male"].values
nontarget_scores_male_eval = results_eval.loc[(results_eval['asv_label'] == "nontarget") & (results_eval['its_male'] == True) ]["asv_score_male"].values
spoof_scores_male_eval = results_eval.loc[(results_eval['asv_label'] ==  "spoof") & (results_eval['its_male'] == True)]["asv_score_male"].values

list_asv_score_male = list_asv_score = [0.352]

In [27]:
'LA_E_2820896'

'LA_E_2820896'

In [38]:
import pickle

#asv male

# cm male
score_cm_male = 1-score_eval_male_gender_classification.flatten()
cm_thr_male =  0.88 # from the plot

#cm female
score_cm_female = -1*score_eval_female_gender_classification.flatten()
cm_thr_female =  0.7000000000000015 # from the plot

cm_filenames = np.concatenate((Eval_dataset_all.name[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 1], Eval_dataset_all.name[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 0]))

cm_speakers_sex = np.concatenate((Eval_dataset_all.sex[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 1],Eval_dataset_all.sex[gender_model.predict(Eval_dataset_all.data_for_gender_classification) == 0]))

asv_male_filenames = results_eval.loc[(results_eval['its_male'] == True)]["file_name"].values

asv_female_filenames = results_eval.loc[(results_eval['its_male'] == False)]["file_name"].values

asv_score_male = results_eval.loc[(results_eval['its_male'] == True)]["asv_score_male"].values

asv_score_female = results_eval.loc[(results_eval['its_male'] == False)]["asv_score_female"].values

asv_speakers =  np.concatenate((results_eval.loc[(results_eval['its_male'] == True)]["speaker_id_male"].values,results_eval.loc[(results_eval['its_male'] == False)]["speaker_id_female"].values))

asv_labels = np.concatenate((results_eval.loc[(results_eval['its_male'] == True)]["asv_label"].values,results_eval.loc[(results_eval['its_male'] == False)]["asv_label"].values))

calc_a_dcf_spesific(cm_filenames,cm_speakers_sex,asv_male_filenames,asv_female_filenames,asv_speakers,asv_labels,score_cm_male,asv_score_male,cm_thr_male,score_cm_female,asv_score_female,cm_thr_female)


a-DCF: 0.12033, threshold: 0.36817
Male CM Threshold: 0.88,Female CM Threshold: 0.7000000000000015, 	Min a-dcf: 0.12033351943056518 	Min a-dcf threshold: 0.36816923726688733


(0.12033351943056518,
 0.36816923726688733,
 array([1.        , 1.00029435, 1.0005887 , ..., 1.58008356, 1.58037792,
        1.58067227]),
 array([      -inf,       -inf,       -inf, ..., 0.83300618, 0.83519231,
        0.83914347]))

# Dev

In [29]:
labels_dev_male = torch.Tensor(Dev_dataset_all.is_spoofed[(Dev_dataset_all.sex == 'male').values].values).cpu().numpy().copy()
total_data = torch.Tensor(Dev_dataset_all.data[(Dev_dataset_all.sex == 'male').values]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    score_dev_male = torch.sigmoid(spoof_model_male(torch.Tensor(total_data).to(device)))
    
score_dev_male = score_dev_male.cpu().numpy().copy()

dev_male_eer, _ = my_functions.compute_eer(labels_dev_male,score_dev_male) # compute equal error rate

print(f"\tDev male EER: ({100*dev_male_eer}%)")


	Dev male EER: (0.7981601731367872%)


In [30]:
labels_dev_female = torch.Tensor(Dev_dataset_all.is_spoofed[(Dev_dataset_all.sex == 'female').values].values).cpu().numpy().copy()
total_data = torch.Tensor(Dev_dataset_all.data[(Dev_dataset_all.sex == 'female').values]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    output = spoof_model_female(torch.Tensor(total_data).to(device))
    _ , score_dev_female = spoof_model_female.loss(torch.Tensor(output).to(device),None)
    score_dev_female = -1*score_dev_female
    
score_dev_female = score_dev_female.cpu().numpy().copy()

eer, thresh = my_functions.compute_eer(labels_dev_female,score_dev_female) # compute equal error rate

print(f"\tDev Female EER:({100*eer}%)")

	Dev Female EER:(0.0%)


c:\Users\avish\OneDrive\Desktop\thesis_research\avishai_work\env\lib\site-packages\scipy\interpolate\_interpolate.py:653: RuntimeWarning: divide by zero encountered in true_divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
c:\Users\avish\OneDrive\Desktop\thesis_research\avishai_work\env\lib\site-packages\scipy\interpolate\_interpolate.py:656: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo


In [31]:
from ASV_utils.tdcf_functions import compute_t_dcf


import pickle
results = pickle.load(open("./dcf_and_tdcf\dcf_avg_score/pmf_both_tdcf/asv_protocol/objects_asv_and_cm_time_embeddings_avg_score_both_asv_protocol.pkl", 'rb'))

# Dev with Labels / Preds (is the same):

In [32]:
bonafide_score_cm_male_dev = 1 - score_dev_male[labels_dev_male==0].flatten()
spoof_score_cm_male_dev = 1 - score_dev_male[labels_dev_male==1].flatten()

target_scores_male_dev = results.loc[(results['asv_label'] ==  "target") & (results['its_male_ground_truth'] == 'male') ]["asv_score_male"].values
nontarget_scores_male_dev = results.loc[(results['asv_label'] == "nontarget") & (results['its_male_ground_truth'] == 'male') ]["asv_score_male"].values
spoof_scores_male_dev = results.loc[(results['asv_label'] ==  "spoof") & (results['its_male_ground_truth'] == 'male')]["asv_score_male"].values
list_asv_score_male = [0.261]



bonafide_score_cm_female_dev = score_dev_female[labels_dev_female==0]
bonafide_score_cm_female_dev = -1*bonafide_score_cm_female_dev
spoof_score_cm_female_dev = score_dev_female[labels_dev_female==1]
spoof_score_cm_female_dev = -1*spoof_score_cm_female_dev

target_scores_female_dev = results.loc[(results['asv_label'] ==  "target") & (results['its_male_ground_truth'] == 'female') ]["asv_score_female"].values
nontarget_scores_female_dev = results.loc[(results['asv_label'] == "nontarget") & (results['its_male_ground_truth'] == 'female') ]["asv_score_female"].values
spoof_scores_female_dev = results.loc[(results['asv_label'] ==  "spoof") & (results['its_male_ground_truth'] == 'female')]["asv_score_female"].values
list_asv_score_female = [0.397]


In [33]:
#asv male
target_scores_male = target_scores_male_dev
nontarget_scores_male = nontarget_scores_male_dev
spoof_scores_male = spoof_scores_male_dev 
asv_score_male = list_asv_score_male[0]
# cm male
bonafide_score_cm_male = bonafide_score_cm_male_dev
spoof_score_cm_male = spoof_score_cm_male_dev
thr_male =  0.7655534744262695

#asv female
target_scores_female = target_scores_female_dev 
nontarget_scores_female = nontarget_scores_female_dev
spoof_scores_female = spoof_scores_female_dev 
asv_score_female = list_asv_score_female[0]
# cm female
bonafide_score_cm_female = bonafide_score_cm_female_dev
spoof_score_cm_female = spoof_score_cm_female_dev
thr_female = 0.5543830394744873


_ ,_,_  = compute_t_dcf(bonafide_score_cm_male,spoof_score_cm_male,Prior_spoof,target_scores_male,nontarget_scores_male,spoof_scores_male,[asv_score_male],type)


_ ,_,_  = compute_t_dcf(bonafide_score_cm_female,spoof_score_cm_female,Prior_spoof,target_scores_female,nontarget_scores_female,spoof_scores_female,[asv_score_female],type)


The t-DCF evaluation type is: constrained
t-DCF evaluation from [Nbona=868, Nspoof=7392] trials

t-DCF MODEL
   Ptar         =  0.94050 (Prior probability of target user)
   Pnon         =  0.00950 (Prior probability of nontarget user)
   Pspoof       =  0.05000 (Prior probability of spoofing attack)
   Cfa_asv      = 10.00000 (Cost of ASV falsely accepting a nontarget)
   Cmiss_asv    =  1.00000 (Cost of ASV falsely rejecting target speaker)
   Cfa_cm       = 10.00000 (Cost of CM falsely passing a spoof to ASV system)
   Cmiss_cm     =  1.00000 (Cost of CM falsely blocking target utterance which never reaches ASV)

   Implied normalized t-DCF function (depends on t-DCF parameters and ASV errors), s=CM threshold)
   tDCF_norm(s) =  3.08466 x Pmiss_cm(s) + Pfa_cm(s)

the asv threshold is from eer on asv development set  0.261
the CM thresholds is:  [3.23759556e-04 1.32375956e-03 1.48350000e-03 ... 9.96420860e-01
 9.96460497e-01 9.96598303e-01]
the CM threshold min is: 0.7655534744262695

In [34]:
import pickle

# cm male
score_cm_male = 1-score_dev_male.flatten()
cm_thr_male =   0.76 # validation thr

#cm female
score_cm_female = -1*score_dev_female.flatten()
cm_thr_female =   0.5100000000000013 # validation thr

cm_filenames = np.concatenate((Dev_dataset_all.name[Dev_dataset_all.sex == 'male'], Dev_dataset_all.name[Dev_dataset_all.sex == 'female']))

cm_speakers_sex = np.concatenate((Dev_dataset_all.sex[Dev_dataset_all.sex == 'male'].values, Dev_dataset_all.sex[Dev_dataset_all.sex == 'female'].values))

asv_male_filenames = results.loc[(results['its_male_ground_truth'] == 'male')]["file_name"].values

asv_female_filenames = results.loc[(results['its_male_ground_truth'] == 'female')]["file_name"].values

asv_score_male = results.loc[(results['its_male_ground_truth'] == 'male')]["asv_score_male"].values

asv_score_female = results.loc[(results['its_male_ground_truth'] == 'female')]["asv_score_female"].values

asv_speakers =  np.concatenate((results.loc[(results['its_male_ground_truth'] == 'male')]["speaker_id_male"].values,results.loc[(results['its_male_ground_truth'] == 'female')]["speaker_id_male"].values))

asv_labels = np.concatenate((results.loc[(results['its_male_ground_truth'] == 'male')]["asv_label"].values,results.loc[(results['its_male_ground_truth'] == 'female')]["asv_label"].values))

calc_a_dcf_spesific(cm_filenames,cm_speakers_sex,asv_male_filenames,asv_female_filenames,asv_speakers,asv_labels,score_cm_male,asv_score_male,cm_thr_male,score_cm_female,asv_score_female,cm_thr_female)


a-DCF: 0.00635, threshold: 0.28467
Male CM Threshold: 0.76,Female CM Threshold: 0.5100000000000013, 	Min a-dcf: 0.006348561573100347 	Min a-dcf threshold: 0.28467178815289546


(0.006348561573100347,
 0.28467178815289546,
 array([1.        , 1.00106514, 1.00102745, ..., 1.57854198, 1.57960713,
        1.58067227]),
 array([      -inf,       -inf,       -inf, ..., 0.83441385, 0.83831373,
        0.84126422]))

# Dev with Preds:

In [35]:
labels_dev_male_gender_classification = torch.Tensor(Dev_dataset_all.is_spoofed[gender_model.predict(Dev_dataset_all.data_for_gender_classification) == 1]).cpu().numpy().copy()
total_data = torch.Tensor(Dev_dataset_all.data[gender_model.predict(Dev_dataset_all.data_for_gender_classification) == 1]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    score_dev_male_gender_classification = torch.sigmoid(spoof_model_male(torch.Tensor(total_data).to(device)))
    
score_dev_male_gender_classification = score_dev_male_gender_classification.cpu().numpy().copy()

dev_male_eer, _ = my_functions.compute_eer(labels_dev_male_gender_classification,score_dev_male_gender_classification) # compute equal error rate

print(f"\tDev male EER: ({100*dev_male_eer}%)")



	Dev male EER: (0.7981601731367872%)


In [36]:
labels_dev_female_gender_classification = torch.Tensor(Dev_dataset_all.is_spoofed[gender_model.predict(Dev_dataset_all.data_for_gender_classification) == 0].values).cpu().numpy().copy()
total_data = torch.Tensor(Dev_dataset_all.data[gender_model.predict(Dev_dataset_all.data_for_gender_classification) == 0]).cpu().numpy().copy()
with torch.no_grad():
    model = model.to(device)
    output = spoof_model_female(torch.Tensor(total_data).to(device))
    _ , score_dev_female_gender_classification = spoof_model_female.loss(torch.Tensor(output).to(device),None)
    score_dev_female_gender_classification = -1*score_dev_female_gender_classification
    
score_dev_female_gender_classification = score_dev_female_gender_classification.cpu().numpy().copy()

eer, thresh = my_functions.compute_eer(labels_dev_female_gender_classification,score_dev_female_gender_classification) # compute equal error rate

print(f"\tDev Female EER:({100*eer}%)")

	Dev Female EER:(0.0%)


c:\Users\avish\OneDrive\Desktop\thesis_research\avishai_work\env\lib\site-packages\scipy\interpolate\_interpolate.py:653: RuntimeWarning: divide by zero encountered in true_divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
c:\Users\avish\OneDrive\Desktop\thesis_research\avishai_work\env\lib\site-packages\scipy\interpolate\_interpolate.py:656: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo


In [37]:
bonafide_score_cm_male_dev = 1-score_dev_male_gender_classification[labels_dev_male_gender_classification==0].flatten()
spoof_score_cm_male_dev = 1-score_dev_male_gender_classification[labels_dev_male_gender_classification==1].flatten()
Prior_spoof = 0.05
target_scores_male_dev = results.loc[(results['asv_label'] ==  "target") & (results['its_male'] == True) ]["asv_score_male"].values
nontarget_scores_male_dev = results.loc[(results['asv_label'] == "nontarget") & (results['its_male'] == True) ]["asv_score_male"].values
spoof_scores_male_dev = results.loc[(results['asv_label'] ==  "spoof") & (results['its_male'] == True)]["asv_score_male"].values



bonafide_score_cm_female_dev = score_dev_female_gender_classification[labels_dev_female_gender_classification==0]
bonafide_score_cm_female_dev = -1*bonafide_score_cm_female_dev

spoof_score_cm_female_dev = score_dev_female_gender_classification[labels_dev_female_gender_classification==1]
spoof_score_cm_female_dev = -1*spoof_score_cm_female_dev

target_scores_female_dev = results.loc[(results['asv_label'] ==  "target") & (results['its_male'] == False) ]["asv_score_female"].values
nontarget_scores_female_dev = results.loc[(results['asv_label'] == "nontarget") & (results['its_male'] == False) ]["asv_score_female"].values
spoof_scores_female_dev = results.loc[(results['asv_label'] ==  "spoof") & (results['its_male'] == False)]["asv_score_female"].values


list_asv_score_female = [0.397]
list_asv_score_male = [0.261]